# Chapter 10 — Broadcasting

Part 4: Broadcasting and tensor manipulation.

**Thread.**

- **Recurring idea.** Broadcasting is an index map with repeated coordinates: the shape rule decides legality, the zero stride makes replication free, and the kernel indexes the short operand instead of copying it.
- **Chain.** Scalar → vector/row/column → the four `[M,N]` pairings → 3-D and multiple axes → incompatible shapes → zero strides → no-copy fusion.
- **Ladder.** Shape rule and host reference (Method 01) → Triton tile kernels with explicit indexing (Method 02) → zero-stride views and fused per-channel affine (Method 03).

**Study tip.**

- Right-align `(3,4)` and `(4,)` as `3 4` over `1 4`; compare `4 = 4`, then `3` vs `1`.
- Right-align `(3,4)` and `(3,1)` as `3 4` over `3 1`; compare `4` vs `1`, then `3 = 3`.
- Do `(3,4)+(4,3)` by hand: the rightmost `4` vs `3` fails first.

**Learning objectives**

- State and apply the broadcast-shape rule: right-align, pad missing axes with `1`, and require every axis to be equal or `1`.
- Read `[M,N]+[N]`, `[M,N]+[M,1]`, `[M,N]+[1,N]`, `[M,N]+scalar`, and 3-D broadcasts as **address maps**, not copies.
- Explain how a zero stride encodes logical replication, and compute the traffic saved by not materializing a broadcast operand.
- Write Triton kernels that broadcast a row, a column, a scalar, and a 3-D bias with explicit indexing and masks.
- Name the first axis that makes an incompatible pair fail, rather than guessing from the error text.
- Fuse a per-channel affine transform $y=x\,s+b$ without creating `[M,N]` copies of `s` or `b`.

[Index](../../00_index.ipynb) · [← Previous](../../part3_math/ch09_activations/09_complete.ipynb) · [Next →](../ch11_index_transforms/11_complete.ipynb)

**Estimated time:** 60–75 min · **Difficulty:** Intermediate

**Prerequisites:** Chapter 9 (activation functions), Chapter 6 (indexing and offsets), and enough NumPy to have seen the word *broadcast*. No prior GPU experience is assumed.

**Book apparatus:** [Setup](../../docs/SETUP.md) · [Notation](../../docs/NOTATION.md) · [Glossary](../../docs/GLOSSARY.md) · [API index](../../docs/API_INDEX.md) · [Formula index](../../docs/FORMULA_INDEX.md) · [Diagrams](../../docs/DIAGRAMS.md) · [Troubleshooting](../../docs/TROUBLESHOOTING.md) · [Exercise hints](../../docs/SOLUTIONS.md)

## Concept diagrams

**Right alignment.**

- Missing axes are inserted on the **left**, then each pair of axes is compared.
- `(3,4)` with `(4,)` becomes `(3,4)` with `(1,4)`; the result is `(3,4)`.
- `(3,4)` with `(3,1)` stays `(3,4)` with `(3,1)`; the result is `(3,4)`.

```text
  X :  3  4          b :     4          c :  3  1
       3  4               1  4               3  1
       ----              ----               ----
  axis -2: 3 vs 1 -> 3                axis -2: 3 vs 3 -> 3
  axis -1: 4 vs 4 -> 4                axis -1: 4 vs 1 -> 4
  result:  3  4          result: 3  4          result: 3  4
```

**Size-1 expansion.**

- A row `[1,N]` repeats down the rows; a column `[M,1]` repeats across the columns.
- The repeated values share storage; only the index changes.

```text
row b = (b0,b1,b2,b3)            column c = (c0,c1,c2)^T

+----+----+----+----+            +----+
| b0 | b1 | b2 | b3 |            | c0 |  -> c0 in every column
+----+----+----+----+            +----+
| b0 | b1 | b2 | b3 |            | c1 |
+----+----+----+----+            +----+
| b0 | b1 | b2 | b3 |            | c2 |
+----+----+----+----+            +----+
```

**Zero-stride view.**

- A broadcast axis has stride `0`: every logical index on that axis reads the same physical element.
- The address offset on that axis is always `0`.

```text
storage:   b0  b1  b2  b3          (4 floats, 16 bytes)

view [3,4] with stride (0,1):
  row 0:   b0  b1  b2  b3
  row 1:   b0  b1  b2  b3      <- address offset 1*0 = 0
  row 2:   b0  b1  b2  b3
```

**View versus copy.**

```text
b.expand(3,4)                 b.expand(3,4).contiguous()
shape  (3,4)                  shape  (3,4)
stride (0,1)                  stride (4,1)
storage 4 floats              storage 12 floats
shares b's memory             new allocation
```

- **View.** Free, zero strides, shares the base storage.
- **Copy.** One read and one write of the full broadcast shape, then the kernel reads the copy back.

In [ ]:
import sys
from pathlib import Path

root = Path.cwd()
if (root / "00_Basic").exists():
    root = root / "00_Basic"
while root.name != "00_Basic" and root.parent != root:
    root = root.parent
sys.path.insert(0, str(root))

import torch
import triton
import triton.language as tl

from ttools import DEVICE, HAS_GPU, check, require_gpu, run_if_gpu

print("device:", DEVICE, "| GPU:", HAS_GPU, "| triton:", triton.__version__)

# Part I — Mathematics

**Two halves.**

- **Shape rule.** Decides whether two operands may meet.
- **Address rule.** Says which element each output lane reads when they do.

**Notation.**

- Let the padded shapes be $a=(a_0,\dots,a_{d-1})$ and $b=(b_0,\dots,b_{d-1})$, with `1` inserted on the left until the ranks match.
- **Result shape.** For each axis, $r_k=a_k$ when $b_k=1$; $r_k=b_k$ when $a_k=1$; $r_k=a_k$ when $a_k=b_k$; otherwise error.
- **Index map.** $\pi_k(i_k)=0$ when $a_k=1$, else $i_k$; $\tilde a_{i_0,\dots,i_{d-1}}=a_{\pi_0(i_0),\dots,\pi_{d-1}(i_{d-1})}$; $Y_{\mathbf i}=\tilde a_{\mathbf i}\star\tilde b_{\mathbf i}$.

**Consequences.**

- Broadcasting is **logical**: no element of $a$ or $b$ is duplicated in storage.
- A broadcast axis is exactly a **zero stride** in the expanded view, because $\pi_k$ ignores $i_k$ and always reads position `0`.

**Chapter spine.**

| object | shape | values |
|---|---|---|
| $X$ | $(3,4)$ | rows `1,2,3,4`; `5,6,7,8`; `9,10,11,12` |
| $r$ | $(4,)$ | `10,20,30,40` |
| $c$ | $(3,1)$ | `100`; `200`; `300` |
| $s$ | $()$ | `5` |

### 10.1 Scalar broadcasting

**Connection.** Builds on Chapter 7's elementwise contract: a rank-zero operand is the simplest possible broadcast.

- **Rule.** A scalar $s$ has shape $()$, pads to $(1,1)$, and expands on every axis.
- **Formula.** $Y_{ij}=X_{ij}+s$.
- **Example.** With $X=\begin{bmatrix}1&2\\3&4\end{bmatrix}$ and $s=5$: $X+s=\begin{bmatrix}6&7\\8&9\end{bmatrix}$.
- **Where.** Learning-rate scaling, residual shifts, logit temperatures, and the bias of an affine map.
- **Triton.** `x + s` broadcasts inside the tile; for a tensor scalar pass `.item()`, or pass a pointer and load `tl.load(ptr)` once. A length-1 tensor is broadcastable on the host but is not the same object as a scalar.
- **Failure.** A Python list `[5.0]` fails at argument binding; an integer scalar can change promotion, so make the dtype explicit.
- **Bridge.** §10.2 raises the operand to one dimension.

### 10.2 Vector broadcasting

**Connection.** Builds on §10.1 by replacing the scalar with a one-dimensional tile.

- **Rule.** A `[N]` operand aligns with the **last** axis; leading axes are implicit `1`s.
- **Formula.** $[N]+[N]\to[N]$, $[M,N]+[N]\to[M,N]$, and $[B,M,N]+[N]\to[B,M,N]$.
- **Example.** $v=(1,2,3)$ and $w=(10,20,30)$ give $v+w=(11,22,33)$; against a $[2,3]$ matrix, $w$ is treated as $[1,3]$.
- **Where.** Per-feature vectors: a bias, a scale, a mask, or a class prior. This is also the most common shape bug, because the vector attaches to the **columns**.
- **Triton.** `tl.arange(0,N)` has shape `[N]`; write `v[None, :]` before adding it to an `[M,N]` tile.
- **Failure.** A length-$M$ vector with an `[M,N]` matrix is not a column: if $M\ne N$ the host raises; if $M=N$ it silently adds along the columns.
- **Bridge.** §10.3 makes the row axis explicit as `[1,N]`.

In [ ]:
X = torch.tensor([[1.0, 2.0, 3.0, 4.0],
                  [5.0, 6.0, 7.0, 8.0],
                  [9.0, 10.0, 11.0, 12.0]])
r = torch.tensor([10.0, 20.0, 30.0, 40.0])

# 10.1 Scalar: shape () pads to (1,1) and expands on every axis.
assert X.shape == (3, 4) and r.shape == (4,)
assert torch.broadcast_shapes((), X.shape) == (3, 4)
assert (X + 7.0).tolist() == [[8.0, 9.0, 10.0, 11.0],
                              [12.0, 13.0, 14.0, 15.0],
                              [16.0, 17.0, 18.0, 19.0]]

# 10.2 A [N] operand aligns with the last axis of any rank.
assert torch.broadcast_shapes((4,), (4,)) == (4,)
assert torch.broadcast_shapes((2, 3, 4), (4,)) == (2, 3, 4)
assert (r + torch.tensor([1.0, 2.0, 3.0, 4.0])).tolist() == [11.0, 22.0, 33.0, 44.0]
assert (X + r).tolist() == [[11.0, 22.0, 33.0, 44.0],
                            [15.0, 26.0, 37.0, 48.0],
                            [19.0, 30.0, 41.0, 52.0]]
assert (torch.tensor([[1.0, 2.0], [3.0, 4.0]]) + 5).tolist() == [[6.0, 7.0], [8.0, 9.0]]
assert (torch.tensor([1.0, 2.0, 3.0]) + torch.tensor([10.0, 20.0, 30.0])).tolist() == [11.0, 22.0, 33.0]

### 10.3 Row broadcasting

**Connection.** Builds on §10.2's right-aligned vector by promoting it to an explicit `[1,N]` row.

- **Rule.** A row $[1,N]$ is already rank two; its first axis is size `1`, so it expands across the $M$ rows.
- **Formula.** $Y_{ij}=X_{ij}+r_j$ with $r\in\mathbb R^{1\times N}$.
- **Example.** With $X=\begin{bmatrix}1&2&3\\4&5&6\end{bmatrix}$ and $r=\begin{bmatrix}10&20&30\end{bmatrix}$: $X+r=\begin{bmatrix}11&22&33\\14&25&36\end{bmatrix}$.
- **Where.** Per-column bias in a linear layer, the affine parameter of LayerNorm, and the class bias of a classifier.
- **Triton.** Load the row with the column offsets and add it as `r[None, :]`; the load is one vector of length $N$ per tile.
- **Failure.** `r[:, None]` turns the row into a column; the shape may still broadcast, but the values repeat along the wrong axis.
- **Bridge.** §10.4 moves the size-1 axis to the trailing position.

### 10.4 Column broadcasting

**Connection.** Builds on §10.3 by swapping which axis is size one: the vector now carries one value per row.

- **Rule.** A column `[M,1]` has a size-1 last axis, so it expands across the $N$ columns.
- **Formula.** $Y_{ij}=X_{ij}+c_i$ with $c\in\mathbb R^{M\times 1}$.
- **Example.** With $X=\begin{bmatrix}1&2&3\\4&5&6\end{bmatrix}$ and $c=\begin{bmatrix}10\\20\end{bmatrix}$: $X+c=\begin{bmatrix}11&12&13\\24&25&26\end{bmatrix}$.
- **Where.** Per-token scale, per-example weight, row mask, and attention row score; the vector is indexed by the **row** coordinate.
- **Triton.** Load the column with the row offsets and add it as `c[:, None]`; the load is one vector of length $M$ per tile.
- **Failure.** A shape-`[M]` tensor is not a column; it aligns last, so `X + c` works only when $M=N$ and then means something different. Keep `keepdim=True` or `[:, None]`.
- **Bridge.** §10.5 returns to the `[N]` form and reads it as a matrix operation.

In [ ]:
c = torch.tensor([[100.0], [200.0], [300.0]])

# 10.3 Row vector [1,N] expands across rows.
assert torch.broadcast_shapes(X.shape, (1, 4)) == (3, 4)
assert (X + r.reshape(1, 4)).tolist() == [[11.0, 22.0, 33.0, 44.0],
                                          [15.0, 26.0, 37.0, 48.0],
                                          [19.0, 30.0, 41.0, 52.0]]

# 10.4 Column vector [M,1] expands across columns.
assert c.shape == (3, 1)
assert torch.broadcast_shapes(X.shape, c.shape) == (3, 4)
assert (X + c).tolist() == [[101.0, 102.0, 103.0, 104.0],
                            [205.0, 206.0, 207.0, 208.0],
                            [309.0, 310.0, 311.0, 312.0]]
assert (X + c).stride() == (4, 1)

small = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
assert (small + torch.tensor([[10.0, 20.0, 30.0]])).tolist() == [[11.0, 22.0, 33.0],
                                                                  [14.0, 25.0, 36.0]]
assert (small + torch.tensor([[10.0], [20.0]])).tolist() == [[11.0, 12.0, 13.0],
                                                             [24.0, 25.0, 26.0]]

### 10.5 `[M,N] + [N]`

**Connection.** Builds on §10.2's right-aligned `[N]` and §10.3's row form: the vector now meets a matrix.

- **Rule.** Pad the vector to `[1,N]`, then compare axes from the right.
- **Formula.** $X:(M,N)$, $b:(1,N)$, result $(M,N)$; $Y_{ij}=X_{ij}+b_j$.
- **Example.** For $M=3,N=4$: axis $-1$ is $4=4$; axis $-2$ is $3$ vs $1$, so the result is $(3,4)$. With the spine, $X+r=\begin{bmatrix}11&22&33&44\\15&26&37&48\\19&30&41&52\end{bmatrix}$.
- **Where.** Bias add after a matrix multiply, and the per-channel affine of a convolution written as a matrix.
- **Triton.** Use a 2-D grid over `(rows, cols)`; index $X$ with `rows*N + cols` and $b$ with `cols`. The bias load has length `BLOCK_N`, not `BLOCK_M*BLOCK_N`.
- **Failure.** Flattening $X$ and using `offs` instead of `offs % N`: the bug disappears when the tile is one row and appears when `BLOCK` spans more than one row.
- **Bridge.** §10.6 keeps the trailing `1` instead.

### 10.6 `[M,N] + [M,1]`

**Connection.** Builds on §10.5 by moving the size-1 axis from the row position to the column position.

- **Rule.** The trailing size-1 axis expands; the leading axis matches.
- **Formula.** $X:(M,N)$, $c:(M,1)$, result $(M,N)$; $Y_{ij}=X_{ij}+c_i$.
- **Example.** For $M=3,N=4$: axis $-1$ is $4$ vs $1$, giving $4$; axis $-2$ is $3=3$. With the spine, $X+c=\begin{bmatrix}101&102&103&104\\205&206&207&208\\309&310&311&312\end{bmatrix}$.
- **Where.** Example weighting, row normalization, and the row mask of an attention matrix; it is the transpose of row bias in the index sense.
- **Triton.** Load $c$ with the row offsets and write `c[:, None]`; the tile is `[BLOCK_M,1]` and expands across `[BLOCK_M,BLOCK_N]`.
- **Failure.** `.squeeze()` or `.flatten()` turns `[M,1]` into `[M]`; the new shape aligns last and either raises or, when $M=N$, multiplies the wrong coordinate.
- **Bridge.** §10.7 writes the row axis explicitly without changing the result.

### 10.7 `[M,N] + [1,N]`

**Connection.** Builds on §10.5 and §10.6: this is the row-broadcast shape with the size-1 axis kept explicit.

- **Rule.** The first axis is explicitly size one, so it expands.
- **Formula.** $X:(M,N)$, $b:(1,N)$, result $(M,N)$; $Y_{ij}=X_{ij}+b_j$.
- **Example.** $X+\begin{bmatrix}10&20&30&40\end{bmatrix}=X+r$; the numbers are those of §10.5.
- **Where.** `keepdim=True` preserves `[1,N]` so a later subtraction or division does not align with a different axis; `[N]` has rank one and is padded, `[1,N]` is already rank two.
- **Triton.** `b[None, :]` is exactly the `[1,N]` form; write it explicitly so the kernel shape matches the notes.
- **Failure.** Assuming `[1,N]` and `[N]` are interchangeable in higher rank; against `[B,M,N]` both broadcast across batch and rows, but a `[1,N]` meant as `[M,1]` expands the wrong axis.
- **Bridge.** §10.8 shrinks the operand to a scalar and expands both axes.

### 10.8 `[M,N] + scalar`

**Connection.** Builds on §10.1 and completes the shape ladder: the scalar is the rank-zero limit where every axis is size one.

- **Rule.** A scalar `[1,1]` expands on both axes.
- **Formula.** $X:(M,N)$, $s:(1,1)$, result $(M,N)$; $Y_{ij}=X_{ij}+s$.
- **Example.** With the spine and $s=-2$: $X-2=\begin{bmatrix}-1&0&1&2\\3&4&5&6\\7&8&9&10\end{bmatrix}$.
- **Where.** Optimizers, schedulers, and normalization layers all carry scalars of this form.
- **Triton.** No pointer is needed for a kernel-argument scalar; for a shape-`()` tensor pass `.item()`, so the argument is a value rather than an address.
- **Failure.** Treating a shape-`[1]` tensor as a scalar and indexing it with `cols` reads past the one element; use a true scalar or a length-1 pointer indexed at `0`.
- **Bridge.** §10.9 adds a batch axis and returns to vectors.

In [ ]:
# 10.5 / 10.6 / 10.7 / 10.8: four ways to reach the same (3,4) shape.
assert torch.broadcast_shapes((3, 4), (4,)) == (3, 4)
assert torch.broadcast_shapes((3, 4), (3, 1)) == (3, 4)
assert torch.broadcast_shapes((3, 4), (1, 4)) == (3, 4)
assert torch.broadcast_shapes((3, 4), ()) == (3, 4)

row_result = [[11.0, 22.0, 33.0, 44.0],
              [15.0, 26.0, 37.0, 48.0],
              [19.0, 30.0, 41.0, 52.0]]
col_result = [[101.0, 102.0, 103.0, 104.0],
              [205.0, 206.0, 207.0, 208.0],
              [309.0, 310.0, 311.0, 312.0]]
scalar_result = [[6.0, 7.0, 8.0, 9.0],
                 [10.0, 11.0, 12.0, 13.0],
                 [14.0, 15.0, 16.0, 17.0]]

assert (X + r).tolist() == row_result
assert (X + r.reshape(1, 4)).tolist() == row_result
assert (X + c).tolist() == col_result
assert (X + torch.tensor(5.0)).tolist() == scalar_result
assert (X + 5).tolist() == scalar_result

### 10.9 3-D broadcasting

**Connection.** Builds on §10.5 by adding a batch axis in front of the matrix.

- **Rule.** For $X\in\mathbb R^{B\times M\times N}$ and $b\in\mathbb R^{N}$, pad the vector to `[1,1,N]`.
- **Formula.** $Y_{bij}=X_{bij}+b_j$.
- **Example.** With $B=2,M=2,N=3$, $b=(10,20,30)$, $X_0=\begin{bmatrix}1&2&3\\4&5&6\end{bmatrix}$, and $X_1=\begin{bmatrix}7&8&9\\10&11&12\end{bmatrix}$: $Y_0=\begin{bmatrix}11&22&33\\14&25&36\end{bmatrix}$ and $Y_1=\begin{bmatrix}17&28&39\\20&31&42\end{bmatrix}$.
- **Where.** A per-feature bias shared by every batch element and sequence position; convolutional channel biases and shared scales in normalizing flows.
- **Triton.** Use a 3-D grid `(B, ceil(M/BLOCK_M), ceil(N/BLOCK_N))`; the batch offset is `pid_b*M*N`, the feature offset is `cols`, and $\text{addr}(b,i,j)=\text{base}+b\,MN+i\,N+j$.
- **Failure.** A 2-D grid that forgets the batch stride makes every batch read the first slice; the output shape is still correct, so the bug is silent.
- **Bridge.** §10.10 lets both operands carry different size-1 axes.

### 10.10 Multiple broadcasting dimensions

**Connection.** Builds on §10.9: each operand now supplies a different size-1 axis.

- **Rule.** Both operands may have size-1 axes in different positions; the result takes the non-1 size on every axis.
- **Formula.** $u:(B,1,N)$, $v:(1,M,1)$, result $(B,M,N)$; $Y_{bij}=u_{b,0,j}+v_{0,i,0}$.
- **Example.** Axis $-3$ is $B$ vs $1$, axis $-2$ is $1$ vs $M$, axis $-1$ is $N$ vs $1$. With $B=2,M=3,N=4$, $u_{0,0,:}=(10,20,30,40)$, $u_{1,0,:}=(100,200,300,400)$, and $v_{0,:,0}=(1,2,3)$: $Y_0=\begin{bmatrix}11&21&31&41\\12&22&32&42\\13&23&33&43\end{bmatrix}$ and $Y_1=\begin{bmatrix}101&201&301&401\\102&202&302&402\\103&203&303&403\end{bmatrix}$.
- **Where.** Outer sums, pairwise interaction tables, and positional-plus-channel features.
- **Triton.** The kernel sees pointers, not metadata; launch a 3-D grid and index each operand explicitly, or pass a contiguous `[B,M,N]` tile.
- **Failure.** Assuming every size-1 axis expands to the *largest* size in that position; it expands to the pairwise result, which may come from the other operand.
- **Bridge.** §10.11 states exactly when this comparison fails.

In [ ]:
X3 = torch.arange(24.0, dtype=torch.float32).reshape(2, 3, 4)
b3 = torch.tensor([10.0, 20.0, 30.0, 40.0])

# 10.9: [B,M,N] + [N] pads the vector to [1,1,N].
assert X3.shape == (2, 3, 4) and b3.shape == (4,)
assert torch.broadcast_shapes(X3.shape, b3.shape) == (2, 3, 4)
assert (X3 + b3).shape == (2, 3, 4)
assert (X3 + b3)[0, 0].tolist() == [10.0, 21.0, 32.0, 43.0]
assert (X3 + b3)[1, 2].tolist() == [30.0, 41.0, 52.0, 63.0]

X0 = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
X1 = torch.tensor([[7.0, 8.0, 9.0], [10.0, 11.0, 12.0]])
b_ex = torch.tensor([10.0, 20.0, 30.0])
assert (X0 + b_ex).tolist() == [[11.0, 22.0, 33.0], [14.0, 25.0, 36.0]]
assert (X1 + b_ex).tolist() == [[17.0, 28.0, 39.0], [20.0, 31.0, 42.0]]

# 10.10: [B,1,N] + [1,M,1] -> [B,M,N].
u = torch.tensor([[[10.0, 20.0, 30.0, 40.0]],
                  [[100.0, 200.0, 300.0, 400.0]]])
v = torch.tensor([[[1.0], [2.0], [3.0]]])
assert u.shape == (2, 1, 4) and v.shape == (1, 3, 1)
assert torch.broadcast_shapes(u.shape, v.shape) == (2, 3, 4)
out = u + v
assert out[0, 0].tolist() == [11.0, 21.0, 31.0, 41.0]
assert out[0, 2].tolist() == [13.0, 23.0, 33.0, 43.0]
assert out[1, 2].tolist() == [103.0, 203.0, 303.0, 403.0]

### 10.11 Incompatible shapes

**Connection.** Builds on the shape rule used in §§10.5–10.10 and asks what happens when it cannot be satisfied.

- **Rule.** After right alignment and left padding, every axis must satisfy $a_k=b_k$ or $a_k=1$ or $b_k=1$.
- **Formula.** The first axis from the right that violates the condition is the one to report.
- **Example — row axis.** $(3,4)$ with $(2,4)$: axis $-1$ is $4=4$ and passes; axis $-2$ is $3$ vs $2$ and fails.
- **Example — column axis.** $(3,4)$ with $(4,3)$: axis $-1$ is $4$ vs $3$ and fails.
- **Where.** An explicit error is a feature: silently reinterpreting $(3,4)$ and $(2,4)$ would read unrelated memory and corrupt later results.
- **Triton.** A Triton kernel never raises this error; it receives raw pointers and offsets. Call `torch.broadcast_shapes` on the host before launching.
- **Failure.** "Fixing" the error with `reshape` reorders elements; if a transpose is missing add `.T` or `.permute`, if an axis is missing add `.unsqueeze`.
- **Bridge.** §10.12 shows the zero-stride representation behind a successful broadcast.

In [ ]:
# Valid pairs.
assert torch.broadcast_shapes((3, 4), (4,)) == (3, 4)
assert torch.broadcast_shapes((3, 4), (3, 1)) == (3, 4)

# Invalid pairs: the first failing axis is (3 vs 2) and (4 vs 3), respectively.
for bad in [((3, 4), (2, 4)), ((3, 4), (4, 3)), ((2, 3, 4), (3,))]:
    try:
        torch.broadcast_shapes(*bad)
    except RuntimeError:
        pass
    else:
        raise AssertionError(f"{bad} should not broadcast")

### 10.12 Zero-stride concepts

**Connection.** Builds on §10.11 and §10.2: a legal broadcast is represented by an expanded view whose broadcast axis has stride `0`.

- **Rule.** A stride $s_k$ is the number of elements to advance when the index on axis $k$ increases by one; a broadcast axis does not advance at all.
- **Formula.** $\operatorname{addr}(i,j)=\text{base}+i\,s_i+j\,s_j$.
- **Example.** For the spine, $X$ has shape $(3,4)$, stride $(4,1)$, address $\text{base}+4i+j$, and storage 12 floats; row $r$ has stride $(0,1)$ and address $\text{base}+j$; column $c$ has stride $(1,0)$ and address $\text{base}+i$; scalar $s$ has stride $(0,0)$ and address $\text{base}$.
- **Where.** Zero strides are how `expand` represents broadcasting without a copy; a kernel should load the short operand once per tile, not once per output element.
- **Triton.** Raw pointer arithmetic does not carry a stride-0 axis automatically; load the short operand with `cols` or `rows` and add it as `[1,N]` or `[M,1]`, or pass the scalar as an argument. The zero stride lives in the **index expression**.
- **Failure.** `.view()` on an expanded tensor raises because a zero-stride view is not contiguous; use `reshape` if a copy is acceptable, and `contiguous` if a kernel needs a dense buffer.
- **Bridge.** §10.13 turns the zero-stride view into a traffic argument.

### 10.13 Broadcasting without copies

**Connection.** Builds on §10.12: the zero-stride view is free, so the fused kernel should reuse the short operand instead of materializing it.

- **Rule.** `expand` returns a view; the arithmetic is logically elementwise on the expanded shape, but the broadcast operand still occupies only its original storage.
- **Formula.** For a row bias with $M$ rows, $N$ columns, and $b$ bytes per element: $Q_{\text{fused}}=(2M+1)Nb$ and $Q_{\text{copy}}=(4M+1)Nb$. For a per-channel affine: $Q_{\text{fused}}=(2M+2)Nb$ and $Q_{\text{copy}}=(6M+2)Nb$.
- **Example.** For $M=3,N=4,b=4$: row bias is $112$ B fused vs $208$ B copied; per-channel affine is $128$ B fused vs $320$ B copied, a factor of $2.5$.
- **Where.** Every elementwise kernel in this book is memory-bound; a copy adds traffic without arithmetic and allocates memory proportional to the **expanded** shape, so `bias.expand(B,M,N).contiguous()` can exhaust memory.
- **Triton.** Pass the original short vector and index it with the tile coordinate, as in `bias_ptr + cols`; the kernel then reads $N$ values per tile instead of $MN$.
- **Failure.** Writing through an expanded view aliases every row (and the base vector), and a kernel writing into an expanded output would race with itself. Also beware of an implicit `.contiguous()` in a wrapper.
- **Bridge.** The chapter map and worked example turn these rules into one reference case.

**Where this leads.** Part II writes the host-side reference, Part III moves the index expressions into Triton kernels, and Part IV applies the zero-stride view to a fused per-channel affine.

In [ ]:
# 10.12: zero-stride views.
row_view = r.expand(3, 4)
col_view = c.expand(3, 4)
scalar_view = torch.tensor(5.0).expand(3, 4)

assert X.stride() == (4, 1)
assert row_view.shape == (3, 4) and row_view.stride() == (0, 1)
assert col_view.shape == (3, 4) and col_view.stride() == (1, 0)
assert scalar_view.shape == (3, 4) and scalar_view.stride() == (0, 0)
assert row_view.data_ptr() == r.data_ptr()
assert col_view.data_ptr() == c.data_ptr()
assert row_view.numel() == 12 and r.numel() == 4

# 10.13: a copy changes the stride and the allocation.
dense_row = row_view.contiguous()
assert dense_row.stride() == (4, 1)
assert dense_row.data_ptr() != r.data_ptr()
assert torch.equal(X + row_view, X + dense_row)
assert torch.equal(X + row_view, X + r)

# A write through the view aliases every row; it does not create storage.
alias = torch.tensor([10.0, 20.0, 30.0, 40.0])
alias_view = alias.expand(3, 4)
alias_view[0, 0] = -1.0
assert alias[0].item() == -1.0
assert alias_view[:, 0].tolist() == [-1.0, -1.0, -1.0]
assert alias_view[2, 0].item() == alias[0].item()

## Chapter map

| § | Topic | Rule | Shapes | Kernel/demo |
|---|---|---|---|---|
| **10.1** | Scalar broadcasting | every entry adds $s$ | $[M,N]+()\to[M,N]$ | scalar tile add |
| **10.2** | Vector broadcasting | $[N]$ aligns right | $[N]+[N]\to[N]$ | vector add |
| **10.3** | Row broadcasting | $[1,N]$ expands rows | $[M,N]+[1,N]\to[M,N]$ | row bias kernel |
| **10.4** | Column broadcasting | $[M,1]$ expands columns | $[M,N]+[M,1]\to[M,N]$ | column scale kernel |
| **10.5** | `[M,N] + [N]` | pad $[N]$ to $[1,N]$ | $[M,N]+[N]\to[M,N]$ | row bias kernel |
| **10.6** | `[M,N] + [M,1]` | keep trailing 1 | $[M,N]+[M,1]\to[M,N]$ | column scale kernel |
| **10.7** | `[M,N] + [1,N]` | explicit size-1 row axis | $[M,N]+[1,N]\to[M,N]$ | row bias kernel |
| **10.8** | `[M,N] + scalar` | pad scalar to $[1,1]$ | $[M,N]+()\to[M,N]$ | scalar tile add |
| **10.9** | 3-D broadcasting | batch and row axes expand | $[B,M,N]+[N]\to[B,M,N]$ | 3-D bias kernel |
| **10.10** | Multiple broadcasting dimensions | each size-1 axis expands independently | $[B,1,N]+[1,M,1]\to[B,M,N]$ | 3-D bias kernel |
| **10.11** | Incompatible shapes | equal or one, else error | first bad axis from right | host check |
| **10.12** | Zero-stride concepts | broadcast axis has stride 0 | $(0,1)$, $(1,0)$, $(0,0)$ | `expand` view |
| **10.13** | Broadcasting without copies | view, not allocation | fused traffic $(2M+1)Nb$ | per-channel affine |

**Thread.** The shape rule decides whether the map exists; the stride rule says it is free; the kernel rule says to index the short operand.

## Worked numerical example

**Spine.**

- $X=\begin{bmatrix}1&2&3&4\\5&6&7&8\\9&10&11&12\end{bmatrix}$, $r=\begin{bmatrix}10&20&30&40\end{bmatrix}$.
- $c=\begin{bmatrix}100\\200\\300\end{bmatrix}$, $s=5$.

**Row broadcast.** $r$ is padded to `[1,4]` and repeated on every row:

- $X+r=\begin{bmatrix}11&22&33&44\\15&26&37&48\\19&30&41&52\end{bmatrix}$.

**Column broadcast.** $c$ is already `[3,1]` and repeated on every column:

- $X+c=\begin{bmatrix}101&102&103&104\\205&206&207&208\\309&310&311&312\end{bmatrix}$.

**Scalar broadcast.** $s$ is padded to `[1,1]`:

- $X+s=\begin{bmatrix}6&7&8&9\\10&11&12&13\\14&15&16&17\end{bmatrix}$.

**Explicit row form.** `[M,N] + [1,N]` is numerically identical here:

- $X+\begin{bmatrix}10&20&30&40\end{bmatrix}=X+r$.

**Incompatible pairs.**

- $(3,4)$ with $(2,4)$: axis $-1$ passes ($4=4$); axis $-2$ fails ($3$ vs $2$).
- $(3,4)$ with $(4,3)$: axis $-1$ fails ($4$ vs $3$).

The next cell asserts every matrix above, plus the two failures.

In [ ]:
def test_worked_example():
    '''Verify every matrix in the worked example without printing a table.'''
    Xw = torch.tensor([[1.0, 2.0, 3.0, 4.0],
                       [5.0, 6.0, 7.0, 8.0],
                       [9.0, 10.0, 11.0, 12.0]])
    rw = torch.tensor([10.0, 20.0, 30.0, 40.0])
    cw = torch.tensor([[100.0], [200.0], [300.0]])
    sw = 5.0

    assert (Xw + rw).tolist() == [[11.0, 22.0, 33.0, 44.0],
                                  [15.0, 26.0, 37.0, 48.0],
                                  [19.0, 30.0, 41.0, 52.0]]
    assert (Xw + cw).tolist() == [[101.0, 102.0, 103.0, 104.0],
                                  [205.0, 206.0, 207.0, 208.0],
                                  [309.0, 310.0, 311.0, 312.0]]
    assert (Xw + sw).tolist() == [[6.0, 7.0, 8.0, 9.0],
                                  [10.0, 11.0, 12.0, 13.0],
                                  [14.0, 15.0, 16.0, 17.0]]
    assert torch.equal(Xw + rw.reshape(1, 4), Xw + rw)

    for bad in [((3, 4), (2, 4)), ((3, 4), (4, 3))]:
        try:
            torch.broadcast_shapes(*bad)
        except RuntimeError:
            continue
        raise AssertionError(f"{bad} should fail")
    print("worked example: row, column, scalar, and two failures match")


test_worked_example()

# Part II — Method 01: host-side reference

**Connection.** Builds on Part I's shape rule by executing the definition one scalar at a time.

- **Definition.** Ask PyTorch for the broadcast pair, allocate the result with the broadcast shape, then visit every output lane. It is deliberately slow and explicitly the specification the kernels reproduce.
- **Host call 1.** `torch.broadcast_tensors(a, b)` returns **views** with the broadcast shape; the returned strides show the zero axes directly.
- **Host call 2.** `tensor.expand(shape)` is the one-operand form; it is a view and raises if the requested shape is not broadcast-compatible.
- **Reference contract.** The same code path handles scalars, rows, columns, and 3-D operands; assertions check values, broadcast shapes, and view strides.
- **Bridge.** Part III moves the index expressions into Triton.

In [ ]:
def naive_broadcast(op, a, b):
    '''Method 01: broadcast on the host, then one scalar operation per lane.'''
    ab, bb = torch.broadcast_tensors(torch.as_tensor(a), torch.as_tensor(b))
    out = torch.empty(ab.shape, dtype=ab.dtype, device=ab.device)
    af, bf, of = ab.reshape(-1), bb.reshape(-1), out.reshape(-1)
    for i in range(af.numel()):
        if op == "add":
            of[i] = af[i] + bf[i]
        elif op == "mul":
            of[i] = af[i] * bf[i]
        else:
            raise ValueError(op)
    return out


def test_method01():
    Xw = torch.tensor([[1.0, 2.0, 3.0, 4.0],
                       [5.0, 6.0, 7.0, 8.0],
                       [9.0, 10.0, 11.0, 12.0]])
    rw = torch.tensor([10.0, 20.0, 30.0, 40.0])
    cw = torch.tensor([[100.0], [200.0], [300.0]])
    sw = torch.tensor(5.0)

    assert torch.equal(naive_broadcast("add", Xw, rw), Xw + rw)
    assert torch.equal(naive_broadcast("add", Xw, cw), Xw + cw)
    assert torch.equal(naive_broadcast("add", Xw, sw), Xw + sw)
    assert torch.equal(naive_broadcast("mul", Xw, cw), Xw * cw)

    X3 = torch.arange(24.0).reshape(2, 3, 4)
    b3 = torch.tensor([10.0, 20.0, 30.0, 40.0])
    assert torch.equal(naive_broadcast("add", X3, b3), X3 + b3)

    xb, rb = torch.broadcast_tensors(Xw, rw)
    assert xb.shape == rb.shape == (3, 4)
    assert xb.stride() == (4, 1)
    assert rb.stride() == (0, 1)
    cb = cw.expand(3, 4)
    assert cb.stride() == (1, 0)
    print("method 01: reference values and zero-stride views match")


test_method01()

# Part III — Method 02: Triton tile kernels

**Connection.** Builds on Method 01's views by expressing the same broadcast in index arithmetic.

- **Rule.** Triton sees raw pointers and explicit offsets, not shapes or stride metadata.
- **Row map.** $X[i,j]$ at $iN+j$, and $r[j]$ at $j$.
- **Column map.** $X[i,j]$ at $iN+j$, and $c[i]$ at $i$.
- **Tile map.** For $M=3$, $N=4$, `BLOCK_M=2`, `BLOCK_N=4`, the grid is `(2,1)`:

| `pid_m` | `rows` | row mask | `cols` | col mask | live lanes |
|---:|---|---|---|---|---:|
| 0 | 0,1 | T,T | 0,1,2,3 | T,T,T,T | 8 |
| 1 | 2,3 | T,F | 0,1,2,3 | T,T,T,T | 4 |

- **Mask.** Program 1 has one dead row; the row mask protects the `X` load and the `Y` store, the bias load is guarded by `cols < N`, and masked lanes get the neutral value (`0.0` for addition, `1.0` for multiplication).
- **Pointer walk.** For program 1: row base $2\cdot4=8$, addresses $8+\{0,4\}+j$ for $j=0,1,2,3$; live addresses are $8,9,10,11$ and $12,13,14,15$ is the dead row.
- **Bridge.** Part IV removes the materialized copy at the host boundary.

In [ ]:
@triton.jit
def row_bias_kernel(x_ptr, b_ptr, out_ptr, M, N,
                    BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr):
    pid_m = tl.program_id(0)
    pid_n = tl.program_id(1)
    rows = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    cols = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
    mask = (rows[:, None] < M) & (cols[None, :] < N)

    x = tl.load(x_ptr + rows[:, None] * N + cols[None, :], mask=mask, other=0.0)
    b = tl.load(b_ptr + cols, mask=cols < N, other=0.0)
    y = x + b[None, :]
    tl.store(out_ptr + rows[:, None] * N + cols[None, :], y, mask=mask)


def row_bias(x, b, BLOCK_M=2, BLOCK_N=4):
    '''Method 02: [M,N] + [N], with the vector indexed by the column.'''
    x = torch.as_tensor(x).contiguous()
    b = torch.as_tensor(b).contiguous()
    M, N = x.shape
    out = torch.empty_like(x)
    grid = (triton.cdiv(M, BLOCK_M), triton.cdiv(N, BLOCK_N))
    row_bias_kernel[grid](x, b, out, M, N, BLOCK_M=BLOCK_M, BLOCK_N=BLOCK_N)
    return out


# The tile map for M=3, N=4, BLOCK_M=2, BLOCK_N=4.
tile_rows = torch.arange(4)[:, None].expand(4, 4)
tile_cols = torch.arange(4)[None, :].expand(4, 4)
tile_mask = tile_rows < 3
tile_addr = tile_rows * 4 + tile_cols
assert tile_mask.tolist() == [[True, True, True, True],
                              [True, True, True, True],
                              [True, True, True, True],
                              [False, False, False, False]]
assert tile_addr[2].tolist() == [8, 9, 10, 11]
assert tile_addr[3].tolist() == [12, 13, 14, 15]
assert int(tile_mask.sum()) == 12


def test_row_bias():
    Xw = torch.tensor([[1.0, 2.0, 3.0, 4.0],
                       [5.0, 6.0, 7.0, 8.0],
                       [9.0, 10.0, 11.0, 12.0]], device=DEVICE)
    rw = torch.tensor([10.0, 20.0, 30.0, 40.0], device=DEVICE)
    check("row bias [3,4]+[4]", row_bias(Xw, rw), Xw + rw)

    torch.manual_seed(0)
    x = torch.randn(37, 53, device=DEVICE)
    b = torch.randn(53, device=DEVICE)
    check("row bias ragged", row_bias(x, b, BLOCK_M=8, BLOCK_N=16), x + b)
    print("method 02: row bias indexes the vector by the column")


run_if_gpu(test_row_bias)

In [ ]:
@triton.jit
def col_scale_kernel(x_ptr, c_ptr, out_ptr, M, N,
                     BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr):
    pid_m = tl.program_id(0)
    pid_n = tl.program_id(1)
    rows = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    cols = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
    mask = (rows[:, None] < M) & (cols[None, :] < N)

    x = tl.load(x_ptr + rows[:, None] * N + cols[None, :], mask=mask, other=0.0)
    c = tl.load(c_ptr + rows, mask=rows < M, other=1.0)
    y = x * c[:, None]
    tl.store(out_ptr + rows[:, None] * N + cols[None, :], y, mask=mask)


def col_scale(x, c, BLOCK_M=2, BLOCK_N=4):
    '''Method 02: [M,N] * [M,1], with the vector indexed by the row.'''
    x = torch.as_tensor(x).contiguous()
    c = torch.as_tensor(c).contiguous().reshape(-1)
    M, N = x.shape
    out = torch.empty_like(x)
    grid = (triton.cdiv(M, BLOCK_M), triton.cdiv(N, BLOCK_N))
    col_scale_kernel[grid](x, c, out, M, N, BLOCK_M=BLOCK_M, BLOCK_N=BLOCK_N)
    return out


def test_col_scale():
    Xw = torch.tensor([[1.0, 2.0, 3.0, 4.0],
                       [5.0, 6.0, 7.0, 8.0],
                       [9.0, 10.0, 11.0, 12.0]], device=DEVICE)
    cw = torch.tensor([[2.0], [3.0], [4.0]], device=DEVICE)
    check("column scale [3,4]*[3,1]", col_scale(Xw, cw), Xw * cw)

    torch.manual_seed(0)
    x = torch.randn(37, 53, device=DEVICE)
    c = torch.randn(37, 1, device=DEVICE)
    check("column scale ragged", col_scale(x, c, BLOCK_M=8, BLOCK_N=16), x * c)
    print("method 02: column scale indexes the vector by the row")


run_if_gpu(test_col_scale)

# Part IV — Method 03: broadcasting without copies

**Connection.** Builds on Method 02's indexing rule and on §10.13's zero-stride view.

- **Rule.** At the host boundary `expand` is free and `contiguous()` is not; at the kernel boundary a dense `[M,N]` copy has already paid for the broadcast.
- **Transform.** The applied kernel is the per-channel affine $Y_{ij}=X_{ij}\,s_j+b_j$ with $s,b\in\mathbb R^{N}$.
- **Traffic.** $Q_{\text{fused}}=(2M+2)Nb$ and $Q_{\text{copy}}=(6M+2)Nb$.
- **Copy count.** Materialize $s$ and $b$: each reads $Nb$ and writes $MNb$, total $2(M+1)Nb$; then read $X$, read both copies, and write $Y$ ($4MNb$).
- **Fused count.** One read of each of $X,s,b$ and one write of $Y$.
- **Example.** For $M=3,N=4,b=4$: $Q_{\text{fused}}=128$ B, $Q_{\text{copy}}=320$ B, ratio $2.5$.
- **Where.** The epilogue of a convolution, the scale-and-shift of a normalization layer, and per-channel modulation of a feature map.
- **Check.** The next cell asserts the strides, the sharing, and the exact byte counts before any kernel is launched.
- **Bridge.** The 3-D kernel reuses the same index expression with one more term.

In [ ]:
M, N, itemsize = 3, 4, 4

row_fused = (2 * M + 1) * N * itemsize
row_copy = (4 * M + 1) * N * itemsize
affine_fused = (2 * M + 2) * N * itemsize
affine_copy = (6 * M + 2) * N * itemsize

assert row_fused == 112 and row_copy == 208
assert affine_fused == 128 and affine_copy == 320
assert affine_copy == 2.5 * affine_fused

# The view shares storage; the copy does not.
row_view = r.expand(M, N)
dense_row = row_view.contiguous()
assert row_view.stride() == (0, 1) and row_view.data_ptr() == r.data_ptr()
assert dense_row.stride() == (N, 1) and dense_row.data_ptr() != r.data_ptr()
assert row_view.numel() == M * N and r.numel() == N
print(f"traffic: row {row_fused} vs {row_copy} B; affine {affine_fused} vs {affine_copy} B")

In [ ]:
@triton.jit
def channel_affine_kernel(x_ptr, scale_ptr, bias_ptr, out_ptr, M, N,
                          BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr):
    pid_m = tl.program_id(0)
    pid_n = tl.program_id(1)
    rows = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    cols = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
    mask = (rows[:, None] < M) & (cols[None, :] < N)

    x = tl.load(x_ptr + rows[:, None] * N + cols[None, :], mask=mask, other=0.0)
    scale = tl.load(scale_ptr + cols, mask=cols < N, other=1.0)
    bias = tl.load(bias_ptr + cols, mask=cols < N, other=0.0)
    y = x * scale[None, :] + bias[None, :]
    tl.store(out_ptr + rows[:, None] * N + cols[None, :], y, mask=mask)


def channel_affine(x, scale, bias, BLOCK_M=2, BLOCK_N=4):
    '''Method 03: fused per-channel affine, no [M,N] materialization.'''
    x = torch.as_tensor(x).contiguous()
    scale = torch.as_tensor(scale).contiguous().reshape(-1)
    bias = torch.as_tensor(bias).contiguous().reshape(-1)
    M, N = x.shape
    assert scale.numel() == N and bias.numel() == N
    out = torch.empty_like(x)
    grid = (triton.cdiv(M, BLOCK_M), triton.cdiv(N, BLOCK_N))
    channel_affine_kernel[grid](x, scale, bias, out, M, N,
                                BLOCK_M=BLOCK_M, BLOCK_N=BLOCK_N)
    return out


def test_channel_affine():
    torch.manual_seed(1)
    x = torch.randn(37, 53, device=DEVICE)
    scale = torch.randn(53, device=DEVICE)
    bias = torch.randn(53, device=DEVICE)
    check("per-channel affine", channel_affine(x, scale, bias, BLOCK_M=8, BLOCK_N=16),
          x * scale + bias)
    print("method 03: fused affine reads the two vectors without copies")


run_if_gpu(test_channel_affine)

## 3-D broadcasting in a kernel

**Connection.** Builds on §10.9 and Method 02: the same index expression gains a batch term.

- **Rule.** For $X\in\mathbb R^{B\times M\times N}$ and $b\in\mathbb R^{N}$: $Y_{bij}=X_{bij}+b_j$ and $\text{addr}(b,i,j)=\text{base}+b\,MN+i\,N+j$.
- **Grid.** `pid_b` selects the batch slice, `pid_m` the row tile, and `pid_n` the column tile.
- **Bias.** Loaded once per column tile, so the kernel never materializes a `[B,M,N]` bias.
- **CPU check.** `[B,1,N] + [1,M,1]` uses two different size-1 axes; the result takes $B$ from the first operand, $M$ from the second, and $N$ from the first.
- **Bridge.** The pitfall table and exercises apply these rules to new shapes.

In [ ]:
@triton.jit
def bias3d_kernel(x_ptr, b_ptr, out_ptr, B, M, N,
                  BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr):
    pid_b = tl.program_id(0)
    pid_m = tl.program_id(1)
    pid_n = tl.program_id(2)
    rows = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    cols = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
    mask = (rows[:, None] < M) & (cols[None, :] < N)
    base = pid_b * M * N

    x = tl.load(x_ptr + base + rows[:, None] * N + cols[None, :], mask=mask, other=0.0)
    b = tl.load(b_ptr + cols, mask=cols < N, other=0.0)
    y = x + b[None, :]
    tl.store(out_ptr + base + rows[:, None] * N + cols[None, :], y, mask=mask)


def bias3d(x, b, BLOCK_M=2, BLOCK_N=4):
    '''3-D broadcast: [B,M,N] + [N], one program per batch and tile.'''
    x = torch.as_tensor(x).contiguous()
    b = torch.as_tensor(b).contiguous().reshape(-1)
    B, M, N = x.shape
    out = torch.empty_like(x)
    grid = (B, triton.cdiv(M, BLOCK_M), triton.cdiv(N, BLOCK_N))
    bias3d_kernel[grid](x, b, out, B, M, N, BLOCK_M=BLOCK_M, BLOCK_N=BLOCK_N)
    return out


def test_bias3d():
    X3w = torch.arange(24.0, device=DEVICE).reshape(2, 3, 4)
    b3w = torch.tensor([10.0, 20.0, 30.0, 40.0], device=DEVICE)
    check("3-D [2,3,4]+[4]", bias3d(X3w, b3w), X3w + b3w)

    u = torch.tensor([[[10.0, 20.0, 30.0, 40.0]],
                      [[100.0, 200.0, 300.0, 400.0]]], device=DEVICE)
    v = torch.tensor([[[1.0], [2.0], [3.0]]], device=DEVICE)
    assert torch.broadcast_shapes(u.shape, v.shape) == (2, 3, 4)
    check("multiple dims [2,1,4]+[1,3,1]", u + v,
          u.expand(2, 3, 4) + v.expand(2, 3, 4))
    print("3-D: batch offset and two size-1 axes match the text")


run_if_gpu(test_bias3d)

# Pitfalls, self-check, and summary

## Pitfalls

| Pitfall | What happens | Fix |
|---|---|---|
| **Left alignment** | `(3,4)` with `(3,)` compared from the left looks like `3=3`, then fails or misaligns | right-align; pad on the left with `1` |
| **`(M,)` versus `(N,)`** | a length-$M$ vector attaches to the **columns** of `[M,N]` | use `v[:, None]` for rows, `v[None, :]` for columns |
| **Expanding a non-contiguous view** | `.view()` raises on a zero-stride axis | use `reshape` only when a copy is acceptable, otherwise keep the view |
| **Writing through an expanded view** | one write aliases every row (and the base vector) | write to a fresh tensor; never to `expand` output |
| **Memory blow-up** | `.contiguous()` on `bias.expand(B,M,N)` allocates the full shape | pass the short vector and index it in the kernel |
| **Kernel cannot infer a broadcast** | a raw pointer has no shape metadata; wrong length reads wrong memory | check `torch.broadcast_shapes` on the host and pass explicit offsets |
| **Flattening loses the row** | `offs % N` is missing, so the bias repeats the first row | index with `cols`, not the flat offset |
| **Scalar as a length-1 pointer** | the kernel reads past the one element | pass a true scalar or index element `0` |
| **Dropping `keepdim`** | `[1,N]` becomes `[N]` and attaches to the wrong axis later | keep the dimension through the reduction |
| **Wrong `other` for a masked load** | the dead lane contributes `0` to a product | choose the neutral element of the operation that follows |

## Self-check

1. Right-align `(5,1,7)` and `(3,7)` and write the result shape. Which axes are size-1?
2. Why does `(3,4) + (3,)` fail, while `(3,4) + (4,)` succeeds? Name the axis in each case.
3. For $X$ of shape `(4,6)`, what are the strides of `r.expand(4,6)` and `c.expand(4,6)` for $r$ of shape `(6,)` and $c$ of shape `(4,1)`?
4. Write the address formula for a row broadcast and for a column broadcast. Which term disappears in each?
5. What is the traffic ratio between a fused row bias and a materialized row bias for $M=64$, $N=512$, `float32`?
6. What does `x.expand(3,4)[0, 0] = 1.0` do to the other rows and to the base vector? Why must a kernel never write into an expanded output?
7. In the row-bias kernel, which lanes does the bias load touch? Which lanes does the `X` load touch?
8. Extend the 3-D kernel to `[B,M,N] + [M,1]`. What changes in the pointer expression and in the bias load?

## Summary

- Broadcasting is a shape rule plus an index map; it is not a loop and not a copy.
- Right-align the shapes, pad missing axes with `1`, and require every axis to be equal or `1`.
- A size-1 axis expands to the size of the other operand; its stride is `0`.
- `[N]` attaches to the last axis. A column needs an explicit `[M,1]`.
- A scalar is the rank-zero case and expands on every axis.
- 3-D and multi-axis broadcasts are the same rule with more axes; each size-1 axis expands independently.
- An incompatible pair fails at the first axis from the right that is neither equal nor `1`.
- `expand` is a free view; `contiguous()` is a copy. Fused kernels index the short operand and skip the copy.

**Where this leads.** The ten exercises turn the rules into practice; Chapter 11 reuses this stride reasoning for flatten, transpose, gather, and scatter.

# Exercises

1. **Axis table.** For each pair, write the padded shapes, the comparison on every axis, and the result: `(4,3)+(3,)`, `(4,3)+(4,1)`, `(4,3)+(1,3)`, `(4,3)+()`, `(4,3)+(3,4)`.
2. **First failure.** Implement a helper that returns the first failing axis from the right for two shapes. Check it on `(3,4)+(2,4)` and `(3,4)+(4,3)`.
3. **Vector versus column.** Construct a square matrix where `X + v` and `X + v[:, None]` both run but give different results. Print neither; assert the inequality.
4. **Zero stride.** Use `as_strided` to build the row and column broadcast views of a small vector and assert their strides and values against `expand`.
5. **Traffic.** Derive $Q_{\text{fused}}$ and $Q_{\text{copy}}$ for a row bias with $M=1024$, $N=4096$, `float16`. How many megabytes does the copy add?
6. **Ragged tiles.** Change `BLOCK_M` and `BLOCK_N` in the row-bias kernel so that both $M$ and $N$ have a partial tile. Assert the result against PyTorch.
7. **3-D column.** Extend `bias3d_kernel` to accept a column `[M,1]` as well as a row `[N]`. Which pointer index changes?
8. **Per-channel affine.** Compare the fused kernel with a host version that calls `.contiguous()` on both expanded vectors. Measure the allocation size on a large shape.
9. **Backward.** For $Y=X+r$ with $r$ of shape `[N]`, show that $\partial L/\partial r_j=\sum_i g_{ij}$. Write the sum as a reduction along axis 0 and check it with autograd.
10. **No-copy rule.** Write a wrapper that asserts its bias argument is already a view with a zero stride, and raises if it receives a materialized copy instead.

In [ ]:
print("Running CPU tests ...")
test_worked_example()
test_method01()

print("\nRunning GPU tests (skipped without CUDA/ROCm) ...")
for fn in (test_row_bias, test_col_scale, test_channel_affine, test_bias3d):
    run_if_gpu(fn)